In [5]:
from Bio.PDB import PDBParser, PDBIO, Superimposer, Select, Structure, Model, Chain, Residue, Atom
from Bio.Seq import Seq
from Bio.SeqUtils import seq1
from Bio import pairwise2
import numpy as np
import os


pdb_file = os.path.expanduser('1gc4.pdb')


with open(pdb_file, 'r') as file:
    pdb_lines = file.readlines()
print(f"Количество строк в PDB-файле: {len(pdb_lines)}")

# Ищем индексы начала и конца секции атомо
atom_start = None
for i, line in enumerate(pdb_lines):
    if line.startswith('ATOM') or line.startswith('HETATM'):
        atom_start = i
        break

atom_end = None
for i in range(len(pdb_lines)-1, -1, -1):
    if pdb_lines[i].startswith('ATOM') or pdb_lines[i].startswith('HETATM'):
        atom_end = i
        break

print(f"Начало секции атомов: строка {atom_start}")
print(f"Конец секции атомов: строка {atom_end}")

header_lines = pdb_lines[:atom_start]
atom_lines = pdb_lines[atom_start:atom_end+1]
footer_lines = pdb_lines[atom_end+1:]

# Парсим структуру белка
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', pdb_file)
print(f"Количество моделей в структуре: {len(structure)}")

# Получаем первую модель
model = structure[0]

# Получаем список цепей
chains = list(model.get_chains())
print(f"Количество цепей: {len(chains)}")
for chain in chains:
    print(f"Цепь ID: {chain.id}")

# Извлекаем последовательности аминокислот из цепей
def get_sequence(chain):
    seq = ''
    for residue in chain:
        if residue.id[0] == ' ':
            seq += seq1(residue.resname)
    return Seq(seq)

sequences = {}
for chain in chains:
    sequences[chain.id] = get_sequence(chain)
    print(f"Последовательность цепи {chain.id}: {sequences[chain.id]}")

# Находим группы идентичных цепей
identical_chains = []
chain_ids = [chain.id for chain in chains]
visited = set()

for i in range(len(chains)):
    chain_i = chains[i]
    seq_i = sequences[chain_i.id]
    group = [chain_i.id]
    if chain_i.id in visited:
        continue
    visited.add(chain_i.id)
    for j in range(i+1, len(chains)):
        chain_j = chains[j]
        if chain_j.id in visited:
            continue
        seq_j = sequences[chain_j.id]
        # Выполняем глобальное выравнивание
        alignments = pairwise2.align.globalxx(seq_i, seq_j)
        alignment = alignments[0]
        identity = alignment[2] / alignment[4] * 100
        if identity == 100.0:
            group.append(chain_j.id)
            visited.add(chain_j.id)
    identical_chains.append(group)

print("Группы идентичных цепей:")
for group in identical_chains:
    print(group)

# Создаем новую структуру и модель
new_structure = Structure.Structure('merged_protein')
new_model = Model.Model(0)
new_structure.add(new_model)

# Словарь для хранения позиций лигандов
ligand_positions = {}

# Обрабатываем каждую группу идентичных цепей
for group in identical_chains:
    # Берем первую цепь как представительную
    chain_id = group[0]
    chain = model[chain_id]
    print(f"Добавляем цепь {chain_id} в новую структуру.")
    new_chain = Chain.Chain(chain_id)
    for residue in chain:
        if residue.id[0] == ' ':  # стандартные аминокислоты
            new_chain.add(residue)
        else:
            # Это лиганд
            ligand_key = residue.resname
            if ligand_key not in ligand_positions:
                ligand_positions[ligand_key] = []
            # Собираем позиции атомов лиганда
            atom_positions = np.array([atom.get_coord() for atom in residue])
            ligand_positions[ligand_key].append(atom_positions)
    new_model.add(new_chain)

# Обрабатываем лиганды
for ligand_key, ligand_list in ligand_positions.items():
    # Количество экземпляров
    num_instances = len(ligand_list)
    print(f"Лиганд {ligand_key}: {num_instances} экземпляров")
    # Составляем массив позиций
    positions_stack = np.stack(ligand_list)  # размерность (экземпляры, атомы, 3)
    # Вычисляем медианные позиции
    median_positions = np.median(positions_stack, axis=0)  # размерность (атомы, 3)
    # Берем первый экземпляр в качестве шаблона
    template_residue = None
    for chain in model.get_chains():
        for residue in chain:
            if residue.resname == ligand_key and residue.id[0] != ' ':
                template_residue = residue
                break
        if template_residue:
            break
    # Создаем новый резидуу
    new_residue = Residue.Residue(template_residue.id, template_residue.resname, template_residue.segid)
    # Создаем новые атомы с медианными позициями
    atom_names = [atom.get_name() for atom in template_residue]
    for i, atom_name in enumerate(atom_names):
        atom = template_residue[atom_name]
        coord = median_positions[i]
        new_atom = Atom.Atom(atom_name, coord, atom.get_bfactor(), atom.get_occupancy(), atom.get_altloc(), atom.get_fullname(), atom.get_serial_number(), element=atom.element)
        new_residue.add(new_atom)
    # Добавляем резидуу в цепь 'L' для лигандов
    if 'L' in new_model.child_dict:
        ligand_chain = new_model['L']
    else:
        ligand_chain = Chain.Chain('L')
        new_model.add(ligand_chain)
    ligand_chain.add(new_residue)
    print(f"Лиганд {ligand_key} добавлен в цепь 'L' с медианными позициями.")

# Определяем класс Select для сохранения всех атомов
class SelectAll(Select):
    def accept_atom(self, atom):
        return True

# Сохраняем новую структуру во временный файл
io = PDBIO()
io.set_structure(new_structure)
temp_pdb_file = 'temp_merged_structure.pdb'
io.save(temp_pdb_file, select=SelectAll())
print(f"Временный файл сохранен: {temp_pdb_file}")

# Читаем атомные записи из временного файла
with open(temp_pdb_file, 'r') as file:
    new_atom_lines = file.readlines()

# Удаляем временный файл
os.remove(temp_pdb_file)
print(f"Временный файл удален: {temp_pdb_file}")

# Собираем финальный PDB-файл
merged_pdb_lines = header_lines + new_atom_lines + footer_lines

# Сохраняем объединенный PDB-файл
output_pdb_file = 'merged_structure.pdb'
with open(output_pdb_file, 'w') as file:
    file.writelines(merged_pdb_lines)
print(f"Объединенный PDB-файл сохранен: {output_pdb_file}")

Количество строк в PDB-файле: 12769
Начало секции атомов: строка 811
Конец секции атомов: строка 12702


Количество моделей в структуре: 1
Количество цепей: 4
Цепь ID: A
Цепь ID: B
Цепь ID: C
Цепь ID: D
Последовательность цепи A: MRGLSRRVQAMKPDAVVAVNAKALELRRQGVDLVALTAGEPDFDTPEHVKEAARRALAQGKTKYAPPAGIPELREALAEKFRRENGLSVTPEETIVTVGGSQALFNLFQAILDPGDEVIVLSPYWVSYPEMVRFAGGVVVEVETLPEEGFVPDPERVRRAITPRTKALVVNSPNNPTGAVYPKEVLEALARLAVEHDFYLVSDEIYEHLLYEGEHFSPGRVAPEHTLTVNGAAKAFAMTGWRIGYACGPKEVIKAMASVSRQSTTSPDTIAQWATLEALTNQEASRAFVEMAREAYRRRRDLLLEGLTALGLKAVRPSGAFYVLMDTSPIAPDEVRAAERLLEAGVAVVPGTDFAAFGHVRLSYATSEENLRKALERFARVL
Последовательность цепи B: MRGLSRRVQAMKPDAVVAVNAKALELRRQGVDLVALTAGEPDFDTPEHVKEAARRALAQGKTKYAPPAGIPELREALAEKFRRENGLSVTPEETIVTVGGSQALFNLFQAILDPGDEVIVLSPYWVSYPEMVRFAGGVVVEVETLPEEGFVPDPERVRRAITPRTKALVVNSPNNPTGAVYPKEVLEALARLAVEHDFYLVSDEIYEHLLYEGEHFSPGRVAPEHTLTVNGAAKAFAMTGWRIGYACGPKEVIKAMASVSRQSTTSPDTIAQWATLEALTNQEASRAFVEMAREAYRRRRDLLLEGLTALGLKAVRPSGAFYVLMDTSPIAPDEVRAAERLLEAGVAVVPGTDFAAFGHVRLSYATSEENLRKALERFARVL
Последовательность цепи C: MRGLSRRVQAMKPDAVVAVNAKALELRRQGVDLVALTAGEPDFDTPEHVKEAARR

Количество строк в PDB-файле: 3696
Начало секции атомов: строка 371
Конец секции атомов: строка 3662
Количество моделей в структуре: 1
Количество цепей: 2
Цепь ID: A
Цепь ID: B
Последовательность цепи A: PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMNLPGRWKPKMIGGIGGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF
Последовательность цепи B: PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMNLPGRWKPKMIGGIGGFIKVRQYDQILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF
Цепь A взаимодействует с лигандами: {'HOH', 'CL'}
Цепь B взаимодействует с лигандами: {'HOH', '6EH'}
Группы идентичных цепей с учетом взаимодействий с лигандами:
['A']
['B']
Добавляем цепь A в новую структуру.
Добавляем цепь B в новую структуру.
Лиганд 6EH: 1 экземпляров, взаимодействует с цепями ('CL', 'HOH')
Лиганд 6EH добавлен в цепь A без изменений.
Лиганд CL: 1 экземпляров, взаимодействует с цепями ('CL', 'HOH')
Лиганд CL добавлен в цепь A без изменений.
Лиганд HOH: 89 экземпляров, взаимодействует с цепями ('CL', 'HOH')
Лиганд HOH добавлен в цепь A без измен

In [8]:
from Bio.PDB import PDBParser, PDBIO, Select, Structure, Model, Chain, Residue, Atom, NeighborSearch
from Bio.SeqUtils import seq1
from Bio import pairwise2
import numpy as np
import os

# Указываем путь к вашему PDB-файлу
pdb_file = '1xxa.pdb'

# Парсим структуру белка
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', pdb_file)
model = structure[0]
chains = list(model.get_chains())
print(f"Total chains: {len(chains)}")

# Извлекаем последовательности аминокислот из цепей
def get_sequence(chain):
    seq = ''
    for residue in chain:
        if residue.id[0] == ' ':
            seq += seq1(residue.resname)
    return seq

sequences = {chain.id: get_sequence(chain) for chain in chains}

# Определяем взаимодействия цепей с лигандами
all_atoms = list(model.get_atoms())
ns = NeighborSearch(all_atoms)
interaction_distance = 5.0
chain_ligand_interactions = {}

for chain in chains:
    interacting_ligands = set()
    for residue in chain:
        for atom in residue:
            neighbor_atoms = ns.search(atom.get_coord(), interaction_distance)
            for neighbor in neighbor_atoms:
                neighbor_residue = neighbor.get_parent()
                neighbor_chain = neighbor_residue.get_parent()
                if neighbor_chain.id != chain.id and neighbor_residue.id[0] != ' ':
                    interacting_ligands.add(neighbor_residue.resname)
    chain_ligand_interactions[chain.id] = interacting_ligands

# Находим группы идентичных цепей с учетом взаимодействий с лигандами
identical_chains = []
visited = set()

for i in range(len(chains)):
    chain_i = chains[i]
    seq_i = sequences[chain_i.id]
    ligands_i = chain_ligand_interactions[chain_i.id]
    group = [chain_i.id]
    if chain_i.id in visited:
        continue
    visited.add(chain_i.id)
    for j in range(i+1, len(chains)):
        chain_j = chains[j]
        if chain_j.id in visited:
            continue
        seq_j = sequences[chain_j.id]
        ligands_j = chain_ligand_interactions[chain_j.id]
        alignments = pairwise2.align.globalxx(seq_i, seq_j)
        identity = alignments[0][2] / alignments[0][4] * 100
        if identity == 100.0 and ligands_i == ligands_j:
            group.append(chain_j.id)
            visited.add(chain_j.id)
    identical_chains.append(group)

print("Identical chain groups considering ligand interactions:")
for group in identical_chains:
    print(f"Group: {group}")

# Создаем новую структуру и модель
new_structure = Structure.Structure('merged_protein')
new_model = Model.Model(0)
new_structure.add(new_model)
ligand_positions = {}

# Обрабатываем каждую группу идентичных цепей
for group in identical_chains:
    chain_id = group[0]
    chain = model[chain_id]
    print(f"Adding chain {chain_id} to new structure.")
    new_chain = Chain.Chain(chain_id)
    for residue in chain:
        if residue.id[0] == ' ':
            new_chain.add(residue.copy())
        else:
            ligand_key = (residue.resname, tuple(sorted(chain_ligand_interactions[chain_id])))
            if ligand_key not in ligand_positions:
                ligand_positions[ligand_key] = []
            atom_positions = np.array([atom.get_coord() for atom in residue])
            ligand_positions[ligand_key].append((atom_positions, residue))
    new_model.add(new_chain)

# Обрабатываем лиганды
for ligand_key, ligand_data_list in ligand_positions.items():
    ligand_name, interacting_chains = ligand_key
    num_instances = len(ligand_data_list)
    if num_instances == 1:
        atom_positions, residue = ligand_data_list[0]
        ligand_chain_id = residue.get_parent().id
        if ligand_chain_id in new_model.child_dict:
            ligand_chain = new_model[ligand_chain_id]
        else:
            ligand_chain = Chain.Chain(ligand_chain_id)
            new_model.add(ligand_chain)
        ligand_chain.add(residue.copy())
        print(f"Ligand {ligand_name} added to chain {ligand_chain_id} without changes.")
    else:
        all_interacting_chains = set()
        for atom_positions, residue in ligand_data_list:
            chain_id = residue.get_parent().id
            all_interacting_chains.update(chain_ligand_interactions[chain_id])
        if len(all_interacting_chains) == 1:
            positions_stack = np.stack([data[0] for data in ligand_data_list])
            median_positions = np.median(positions_stack, axis=0)
            template_residue = ligand_data_list[0][1]
            new_residue = Residue.Residue(template_residue.id, template_residue.resname, template_residue.segid)
            atom_names = [atom.get_name() for atom in template_residue]
            for i, atom_name in enumerate(atom_names):
                atom = template_residue[atom_name]
                coord = median_positions[i]
                new_atom = Atom.Atom(atom_name, coord, atom.get_bfactor(), atom.get_occupancy(),
                                     atom.get_altloc(), atom.get_fullname(), atom.get_serial_number(), element=atom.element)
                new_residue.add(new_atom)
            if 'L' in new_model.child_dict:
                ligand_chain = new_model['L']
            else:
                ligand_chain = Chain.Chain('L')
                new_model.add(ligand_chain)
            ligand_chain.add(new_residue)
            print(f"Ligand {ligand_name} merged and added to chain 'L'.")
        else:
            for atom_positions, residue in ligand_data_list:
                ligand_chain_id = residue.get_parent().id
                if ligand_chain_id in new_model.child_dict:
                    ligand_chain = new_model[ligand_chain_id]
                else:
                    ligand_chain = Chain.Chain(ligand_chain_id)
                    new_model.add(ligand_chain)
                ligand_chain.add(residue.copy())
                print(f"Ligand {ligand_name} added to chain {ligand_chain_id} without changes (interface ligand).")

# Сохраняем новую структуру
io = PDBIO()
io.set_structure(new_structure)
output_pdb_file = 'merged_structure.pdb'
io.save(output_pdb_file)
print(f"Merged PDB file saved: {output_pdb_file}")

Total chains: 6
Identical chain groups considering ligand interactions:
Group: ['A', 'D']
Group: ['B']
Group: ['C']
Group: ['E', 'F']
Adding chain A to new structure.
Adding chain B to new structure.
Adding chain C to new structure.
Adding chain E to new structure.
Ligand ARG added to chain A without changes (interface ligand).
Ligand ARG added to chain A without changes (interface ligand).
Ligand ARG added to chain C without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A without changes (interface ligand).
Ligand HOH added to chain A wi